# Inventory ML End-to-End

This notebook demonstrates the freight-cost regression and invoice anomaly-detection workflows.
The database is intentionally kept local and is not committed to GitHub.

In [ ]:
import pandas as pd
from src.data import load_table
from src.freight_model import build_model, FEATURES, TARGET

df = load_table('vendor_invoice')
df['PODate'] = pd.to_datetime(df['PODate'], errors='coerce')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['days_po_to_invoice'] = (df['InvoiceDate'] - df['PODate']).dt.days
df['po_month'] = df['PODate'].dt.month
df['po_day_of_week'] = df['PODate'].dt.dayofweek
df = df.dropna(subset=[TARGET]).sort_values('PODate')
df.head()

In [ ]:
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
model = build_model()
model.fit(train[FEATURES], train[TARGET])
pred = model.predict(test[FEATURES])
print('Chronological holdout rows:', len(test))

## Evaluation
Use MAE, RMSE and R² for freight-cost regression. A chronological split is preferred because it better represents future prediction than a purely random split.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print('MAE:', mean_absolute_error(test[TARGET], pred))
print('RMSE:', mean_squared_error(test[TARGET], pred) ** 0.5)
print('R2:', r2_score(test[TARGET], pred))

## Invoice anomaly detection
Instead of training on a hand-written invoice flag that would leak the target into the features, the project uses Isolation Forest to identify unusual invoices without a supervised fraud label.

In [ ]:
from src.invoice_anomaly import build_detector, FEATURES as ANOMALY_FEATURES

inv = load_table('vendor_invoice').copy()
for c in ['PODate', 'InvoiceDate']:
    inv[c] = pd.to_datetime(inv[c], errors='coerce')
inv['days_po_to_invoice'] = (inv['InvoiceDate'] - inv['PODate']).dt.days
inv['days_to_pay'] = 0
inv['total_brands'] = 1
inv['total_quantity'] = inv['Quantity']
inv['total_dollars'] = inv['Dollars']
inv['avg_receiving_delay'] = 0
inv = inv.dropna(subset=['Dollars', 'Quantity', 'Freight'])
detector = build_detector()
detector.fit(inv[ANOMALY_FEATURES])
inv['is_anomaly'] = detector.predict(inv[ANOMALY_FEATURES]) == -1
inv['anomaly_score'] = -detector.decision_function(inv[ANOMALY_FEATURES])
inv.sort_values('anomaly_score', ascending=False)[['Dollars','Quantity','Freight','is_anomaly','anomaly_score']].head(10)